<a href="https://colab.research.google.com/github/pranavkantgaur/training_materials/blob/master/nuclear_reactor_lec_1_introduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 1: Introduction to Nuclear Reactor Core Design
## Using Curves and Surfaces for Reactor Physics

### Objectives:
1. Introduce the nuclear reactor core design problem
2. Understand transport-depletion equations in reactor physics
3. Explore how curves and surfaces can model reactor behavior
4. Hands-on: Simple 1D flux profile visualization

## The Central Problem: Reactor Core Design for Target Burnup

### Problem Statement:
**Design a nuclear reactor core that achieves a target burnup over 100 days while satisfying practical constraints.**

### Key Concepts:

**Burnup**: Measure of energy extracted from nuclear fuel (typically in MWd/kg or GWd/tU)
- Represents how much of the fuel has been "consumed"
- Higher burnup = more efficient fuel utilization

**Transport-Depletion Coupling**:
1. **Transport Equation**: Determines neutron flux distribution
   - Governs how neutrons move through the reactor
   - Depends on material composition

2. **Depletion Equation**: Tracks nuclide concentrations over time
   - Fuel burns up (U-235 decreases)
   - Fission products accumulate
   - Plutonium builds up (breeding)

These equations are **coupled**: flux affects depletion, and depletion changes material composition, which affects flux!

### Design Constraints (Neutronics Focus):
- Criticality: $k_{eff} \approx 1.0$ (reactor stays critical)
- Power distribution limits (avoid hotspots)
- Fuel enrichment bounds (e.g., < 5% for LEU)
- Reactivity control capability

## How Curves and Surfaces Help

### 1. Temporal Evolution (Curves)
- Nuclide concentrations change over time → parametric curves
- Reactivity vs. time → curve fitting and prediction
- Control: Hermite/Bezier curves define evolution paths

### 2. Spatial Distribution (Curves & Surfaces)
- 1D: Axial flux profile → curve representation
- 2D: Radial flux map → surface representation
- Design parameters: control points for optimization

### 3. Optimization & Acceleration
- Curve derivatives → sensitivity analysis
- $\frac{\partial \phi}{\partial N}$ (flux gradient w.r.t. nuclide concentration)
- Accelerates iterative transport-depletion solution

### Why This Matters:
- Traditional approach: solve transport-depletion at discrete time steps (expensive!)
- Our approach: use smooth curve representations
  - Interpolate between time steps
  - Enable gradient-based optimization
  - Reduce computational cost

## Simplified Transport-Depletion Equations

### 1D Steady-State Diffusion Equation (simplified transport):
$$-D \frac{d^2\phi}{dx^2} + \Sigma_a \phi = \nu \Sigma_f \phi$$

Where:
- $\phi(x)$: neutron flux at position $x$
- $D$: diffusion coefficient
- $\Sigma_a$: macroscopic absorption cross-section
- $\Sigma_f$: macroscopic fission cross-section
- $\nu$: average neutrons per fission

### Depletion Equations (Bateman equations):
$$\frac{dN_i}{dt} = \sum_j (\lambda_j N_j + \sigma_j \phi N_j) - (\lambda_i + \sigma_i \phi) N_i$$

Where:
- $N_i(t)$: concentration of nuclide $i$ at time $t$
- $\lambda_i$: decay constant
- $\sigma_i$: microscopic cross-section

### Key Nuclides (Simplified U-Pu Chain):
- U-235: Fissile fuel
- U-238: Fertile material → Pu-239
- Pu-239: Bred fissile material
- Fission products: Neutron absorbers (poison)

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint, solve_bvp
from scipy.interpolate import interp1d

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Example 1: Simple 1D Flux Profile

Let's solve a simplified 1D reactor with constant material properties.

**Problem Setup**:
- Slab reactor: $0 \leq x \leq L$ (L = 200 cm)
- Vacuum boundary conditions: $\phi(0) = \phi(L) = 0$
- Uniform material properties

In [ ]:
# Reactor parameters
L = 200.0  # reactor length (cm)
D = 1.0    # diffusion coefficient (cm)
Sigma_a = 0.01  # absorption cross-section (1/cm)
nu = 2.5   # neutrons per fission
Sigma_f = 0.0095  # fission cross-section (1/cm)

# Calculate material buckling
k_inf = (nu * Sigma_f) / Sigma_a
B_material_squared = (k_inf - 1) * Sigma_a / D

print(f"Infinite multiplication factor k_inf: {k_inf:.4f}")
print(f"Material buckling squared B_m^2: {B_material_squared:.6f} cm^-2")

# Analytical solution for 1D slab with vacuum BC
# phi(x) = A * sin(B_g * x) where B_g = pi/L (geometric buckling)
x = np.linspace(0, L, 1000)
B_geometric_squared = (np.pi / L)**2
k_eff = k_inf / (1 + (B_geometric_squared * D / Sigma_a))

print(f"\nGeometric buckling squared B_g^2: {B_geometric_squared:.6f} cm^-2")
print(f"Effective multiplication factor k_eff: {k_eff:.4f}")

# Flux profile (normalized)
phi = np.sin(np.pi * x / L)

# Plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(x, phi, 'b-', linewidth=2, label='Neutron Flux')
plt.xlabel('Position x (cm)', fontsize=12)
plt.ylabel('Normalized Flux φ(x)', fontsize=12)
plt.title('1D Reactor Flux Profile', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()

# Power density proportional to Sigma_f * phi
power = Sigma_f * phi
plt.subplot(1, 2, 2)
plt.plot(x, power, 'r-', linewidth=2, label='Power Density')
plt.xlabel('Position x (cm)', fontsize=12)
plt.ylabel('Normalized Power Density', fontsize=12)
plt.title('Power Distribution', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

print(f"\nPeak-to-average flux ratio: {np.max(phi) / np.mean(phi):.4f}")

## Representing Flux as a Curve

The flux profile is inherently a curve! We can represent it using:
- **Analytical form**: $\phi(x) = A \sin(\pi x / L)$
- **Parametric form**: $P(t) = [x(t), \phi(x(t))]$ where $t \in [0, 1]$
- **Control points**: Sample flux at key locations

This representation enables:
1. Smooth interpolation between discrete points
2. Derivative calculations for sensitivity analysis
3. Optimization with curve control points

In [ ]:
# Sample flux at control points
n_control_points = 5
x_control = np.linspace(0, L, n_control_points)
phi_control = np.sin(np.pi * x_control / L)

# Interpolate using different methods
phi_linear = interp1d(x_control, phi_control, kind='linear')
phi_cubic = interp1d(x_control, phi_control, kind='cubic')

# Evaluate on fine grid
x_fine = np.linspace(0, L, 1000)
phi_exact = np.sin(np.pi * x_fine / L)

# Plot comparison
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(x_fine, phi_exact, 'k-', linewidth=2, label='Exact', alpha=0.7)
plt.plot(x_fine, phi_linear(x_fine), 'b--', linewidth=1.5, label='Linear Interpolation')
plt.plot(x_fine, phi_cubic(x_fine), 'r:', linewidth=2, label='Cubic Interpolation')
plt.plot(x_control, phi_control, 'go', markersize=10, label='Control Points')
plt.xlabel('Position x (cm)', fontsize=12)
plt.ylabel('Flux φ(x)', fontsize=12)
plt.title('Flux Curve Representations', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

# Error analysis
plt.subplot(1, 2, 2)
error_linear = np.abs(phi_exact - phi_linear(x_fine))
error_cubic = np.abs(phi_exact - phi_cubic(x_fine))
plt.semilogy(x_fine, error_linear, 'b--', linewidth=1.5, label='Linear Error')
plt.semilogy(x_fine, error_cubic, 'r:', linewidth=2, label='Cubic Error')
plt.xlabel('Position x (cm)', fontsize=12)
plt.ylabel('Absolute Error', fontsize=12)
plt.title('Interpolation Error', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Max linear interpolation error: {np.max(error_linear):.6f}")
print(f"Max cubic interpolation error: {np.max(error_cubic):.6f}")
print(f"\nCubic interpolation is {np.max(error_linear)/np.max(error_cubic):.1f}x more accurate!")

## Example 2: Time-Dependent Nuclide Concentrations

Now let's look at how fuel composition changes over time - this is the **depletion** part!

**Simplified 3-nuclide chain**:
- U-235 → Fission Products (with flux)
- U-238 → Pu-239 (neutron capture)
- Pu-239 → Fission Products (with flux)

We'll see these concentrations trace out curves in time!

In [ ]:
# Simplified depletion model (constant flux approximation)
def depletion_ode(N, t, phi, sigma):
    """
    Simple 3-nuclide depletion chain
    N = [N_U235, N_U238, N_Pu239]
    """
    N_U235, N_U238, N_Pu239 = N
    
    sigma_f_U235 = sigma['U235_fission']
    sigma_c_U238 = sigma['U238_capture']
    sigma_f_Pu239 = sigma['Pu239_fission']
    
    # Depletion rates
    dN_U235_dt = -sigma_f_U235 * phi * N_U235
    dN_U238_dt = -sigma_c_U238 * phi * N_U238
    dN_Pu239_dt = sigma_c_U238 * phi * N_U238 - sigma_f_Pu239 * phi * N_Pu239
    
    return [dN_U235_dt, dN_U238_dt, dN_Pu239_dt]

# Cross-sections (barns converted to cm^2)
barn_to_cm2 = 1e-24
sigma = {
    'U235_fission': 585 * barn_to_cm2,
    'U238_capture': 2.7 * barn_to_cm2,
    'Pu239_fission': 750 * barn_to_cm2
}

# Initial conditions (atoms/barn-cm)
# 4% enrichment in U-235
rho_U = 19.1  # g/cm^3 for uranium metal
N_A = 6.022e23  # Avogadro's number
A_U = 238  # atomic mass
N_total = rho_U * N_A / A_U  # atoms/cm^3
N_total_barn_cm = N_total * 1e-24  # atoms/(barn-cm)

enrichment = 0.04
N0_U235 = enrichment * N_total_barn_cm
N0_U238 = (1 - enrichment) * N_total_barn_cm
N0_Pu239 = 0.0

N0 = [N0_U235, N0_U238, N0_Pu239]

# Time evolution (100 days)
days = 100
seconds_per_day = 86400
t_max = days * seconds_per_day
t = np.linspace(0, t_max, 1000)
t_days = t / seconds_per_day

# Average flux (neutrons/cm^2/s) - typical thermal reactor
phi_avg = 1e14

# Solve depletion equations
solution = odeint(depletion_ode, N0, t, args=(phi_avg, sigma))
N_U235 = solution[:, 0]
N_U238 = solution[:, 1]
N_Pu239 = solution[:, 2]

# Plot nuclide evolution
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(t_days, N_U235/N0_U235, 'b-', linewidth=2, label='U-235')
plt.plot(t_days, N_U238/N0_U238, 'g-', linewidth=2, label='U-238')
plt.plot(t_days, N_Pu239/N0_U235, 'r-', linewidth=2, label='Pu-239 (scaled)')
plt.xlabel('Time (days)', fontsize=12)
plt.ylabel('Normalized Concentration', fontsize=12)
plt.title('Nuclide Concentrations vs Time', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

# Burnup (proportional to U-235 consumed)
burnup = (N0_U235 - N_U235) / N0_U235 * 100  # percentage
plt.subplot(1, 2, 2)
plt.plot(t_days, burnup, 'purple', linewidth=2.5)
plt.xlabel('Time (days)', fontsize=12)
plt.ylabel('Burnup (% of initial U-235)', fontsize=12)
plt.title('Fuel Burnup vs Time', fontsize=14)
plt.grid(True, alpha=0.3)
plt.axhline(y=burnup[-1], color='r', linestyle='--', alpha=0.5, 
            label=f'Final: {burnup[-1]:.2f}%')
plt.legend()

plt.tight_layout()
plt.show()

print(f"Initial U-235 fraction: {N0_U235/N_total_barn_cm*100:.2f}%")
print(f"Final U-235 fraction: {N_U235[-1]/N_total_barn_cm*100:.2f}%")
print(f"Pu-239 built up: {N_Pu239[-1]/N_total_barn_cm*100:.4f}%")
print(f"\nTotal burnup after 100 days: {burnup[-1]:.2f}%")

## Key Insights from Examples

### 1. Spatial Distribution (Example 1)
- Flux profile is a **curve** in space
- Can be represented with control points
- Higher-order interpolation (cubic) much more accurate
- Design problem: choose control points (enrichment zones) to achieve desired flux shape

### 2. Temporal Evolution (Example 2)
- Nuclide concentrations trace **curves** in time
- Smooth, continuous evolution (no discontinuities)
- Curves couple together (U-238 → Pu-239)
- Design problem: achieve target burnup while maintaining criticality

### 3. The Coupling Challenge
- Flux depends on material composition (through cross-sections)
- Material composition changes due to flux
- Must solve iteratively: transport → depletion → transport → ...
- **Our goal**: Use curve representations to accelerate this!

## Preview of Coming Lectures

### Lecture 2: Hermite Curves for Burnup Control
- Model nuclide evolution with prescribed derivatives
- Control reactivity swing over 100 days
- Enforce physical constraints (monotonicity, bounds)

### Lecture 3: Bezier Curves for Flux Optimization
- Use control points to shape flux distribution
- Achieve flux flattening (reduce peak-to-average)
- Optimize power distribution

### Lecture 4: B-Splines for Multi-Zone Cores
- Local control for complex geometries
- Model radial enrichment variations
- Handle discontinuities at zone boundaries

### Lecture 5: Gradient Acceleration
- Compute $\frac{\partial \phi}{\partial N}$ using curve derivatives
- Perturbation theory and sensitivity
- Speed up transport-depletion iterations

### Lecture 6: 2D Surface Representations
- Extend to Bezier/B-spline surfaces
- Full 2D core design optimization
- Integrated gradient-based approach

## Exercises

1. **Flux Profile Analysis**:
   - Modify the reactor length L and observe how k_eff changes
   - Find the critical length where k_eff = 1.0
   - What happens to the flux shape?

2. **Interpolation Study**:
   - Try different numbers of control points (3, 5, 10, 20)
   - Compare linear vs cubic interpolation errors
   - At what point does adding more points not help?

3. **Depletion Sensitivity**:
   - Change the initial enrichment (2%, 4%, 6%)
   - How does this affect the burnup rate?
   - How does Pu-239 buildup change?

4. **Time Step Analysis**:
   - Solve depletion with fewer time points (10, 50, 100, 500)
   - Compare solutions - when is it "converged"?
   - This motivates need for smooth curve representations!

5. **Design Challenge**:
   - Given a target burnup of 3% after 100 days
   - What initial enrichment is needed?
   - What average flux level?
   - (Hint: you'll need to iterate/search)

## Summary

In this lecture, we:
1. ✅ Introduced the nuclear reactor core design problem
2. ✅ Understood basic transport-depletion equations
3. ✅ Saw how flux profiles are curves in space
4. ✅ Observed nuclide concentrations trace curves in time
5. ✅ Recognized the value of curve representations for:
   - Smooth interpolation
   - Derivative calculations
   - Design optimization

**Next lecture**: We'll use **Hermite curves** to model and control the temporal evolution of nuclide concentrations, ensuring smooth burnup progression while maintaining reactor criticality!